In [7]:
import nest_asyncio
nest_asyncio.apply()

In [8]:
from pathlib import Path
from langchain_core.documents import Document
import re  # For timestamp parsing
import os

folder_path = r"C:\Users\sohai\Desktop\LAB\Final_Project\Final_Files"
documents = []

for txt_file in Path(folder_path).glob("**/*.txt"):
    with open(txt_file, "r", encoding="utf-8") as f:
        content = f.read()
    
    # Extract timestamps (WEBVTT format: 00:01:23.456 --> 00:01:25.789)
    timestamps = re.findall(r'(\d{2}:\d{2}:\d{2}\.\d{3}) --> (\d{2}:\d{2}:\d{2}\.\d{3})', content)
    
    doc = Document(
        page_content=content,
        metadata={
            "source": str(txt_file),
            "type": "txt",
            "file_name": txt_file.name,
            "num_segments": len(timestamps),
            "first_timestamp": timestamps[0][0] if timestamps else "00:00:00.000",
            "last_timestamp": timestamps[-1][1] if timestamps else "00:00:00.000",
            "duration_estimate": "Unknown"  # Could calculate
        }
    )
    documents.append(doc)

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter  # ✅ New package

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500,chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

for i, doc in enumerate(chunks):
    doc.metadata["chunk_id"] = i

for doc in chunks:
    doc.metadata["video_id"] = doc.metadata.get("file_name").split("/")[-1].replace(".txt", "")

In [10]:
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from langchain_openai import OpenAIEmbeddings

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index_name = "rag"
if index_name not in pc.list_indexes().names():
    pc.create_index(name=index_name,dimension=1536,metric="cosine",spec=ServerlessSpec(cloud="aws", region="us-east-1"))

index = pc.Index(index_name)
embeddings = OpenAIEmbeddings(api_key=os.getenv("OPENAI_API_KEY"))
vectorstore = PineconeVectorStore(embedding=embeddings, index=index)
vectorstore.add_documents(chunks)

['0fc4a8ee-2253-4662-9639-2d24b145de30',
 '42cdd41b-7cef-4e6f-bd0d-97dd7b6f1922',
 '75eb9a17-cc36-4507-b3e3-3f9aa1e5aba3',
 'a61ecce2-2d4b-4147-b995-d0df00046bba',
 '4aafa688-85b1-4d57-925e-66e2c4b10905',
 'eedece1b-a24b-4934-93fc-6b4a35420c20',
 '17ef0fec-61a3-485b-b9cc-9493c5f4eb9c',
 '7e4dde40-6931-4830-b64a-b32854d6e755',
 '02c58639-1c44-446b-8441-d71da85d7326',
 'aca3bc34-be75-4dbc-8902-b544ed800363',
 '9b25b97a-74e5-4540-9b95-d5ae2f6fb18c',
 '755a0d96-cf70-4cfa-acc3-f73e1c006e35',
 '1aac107d-faeb-4b95-b9fa-37ec470eb295',
 'c0ae1da2-613c-41d4-b11f-bf164daf1cc4',
 '38bb7f60-f4ab-4482-8cd5-26f944c07459',
 '0ff73fb8-ec97-4d51-9401-6b7b88d0939c',
 '93b0edf7-f874-4eba-8415-291ff83472e0',
 '0a24a6ac-9757-4948-a95e-392c9d3a5afc',
 '43366979-2f6f-4480-84d4-83486185727d',
 '9f3b42f2-fce1-4710-90a2-68d89c00858d',
 '8370f0a8-9088-4dd2-9d9f-0a4cb3e46171',
 '51be59e2-df93-40f1-9072-4f578957d840',
 'fac302c3-42cb-4e63-9612-b70e32bc4080',
 'd8a21346-6a00-4a2c-a9a1-4ce9ae3ed813',
 '0ee7d924-f268-

In [ ]:
import os
import uuid
import gradio as gr
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langsmith import Client
import numpy as np
import datetime

# ── LangSmith tracing ─────────────────────────────────────────────────────
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"]    = "ragas-rag-eval"
os.environ["LANGCHAIN_API_KEY"]    = os.getenv("LANGSMITH_API_KEY", "")

# ── 1. LLM & Retriever ────────────────────────────────────────────────────
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=os.getenv("OPENAI_API_KEY"))

# ── 2. PromptTemplate ─────────────────────────────────────────────────────
rag_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are an AI assistant answering questions using a knowledge base.
Use ONLY the information provided in the context below.

Context: {context}

Question: {question}

If the answer is not in the context, say: "I could not find the answer in the provided documents."
Answer:
"""
)

# ── 3. RAG search function ────────────────────────────────────────────────
def search_documents(question: str):
    docs = retriever.invoke(question)
    context_with_metadata = "\n\n".join([
        f"Source: {doc.metadata}\nContent: {doc.page_content}"
        for doc in docs
    ])
    final_prompt = rag_prompt.format(context=context_with_metadata, question=question)
    response = llm.invoke(final_prompt)
    return response.content

# ── 4. Tool ───────────────────────────────────────────────────────────────
@tool
def rag_search(question: str) -> str:
    """Search the vector database and answer questions about neural networks."""
    return search_documents(question)

# ── 5. Memory + Agent ─────────────────────────────────────────────────────
memory = MemorySaver()

agent = create_react_agent(
    model=llm,
    tools=[rag_search],
    checkpointer=memory,
    prompt="You are a helpful AI assistant that answers questions about neural networks "
           "using the rag_search tool. Always use the tool to find answers. "
           "Remember the conversation history and refer to previous answers when relevant."
)

# ── 6. Chat function with memory ──────────────────────────────────────────
def chat(question: str, history: list, session_id: str) -> tuple:
    if not question.strip():
        return history, "", session_id

    config = {"configurable": {"thread_id": session_id}}

    try:
        result = agent.invoke(
            {"messages": [{"role": "user", "content": question}]},
            config=config
        )
        answer = result["messages"][-1].content

        docs = retriever.invoke(question)
        sources = "\n\n".join([
            f"📄 **Chunk {i+1}** — `{doc.metadata.get('source', 'unknown')}`\n"
            f"{doc.page_content[:300]}..."
            for i, doc in enumerate(docs)
        ])
        full_response = f"{answer}\n\n---\n**📚 Retrieved Sources:**\n{sources}"

    except Exception as e:
        full_response = f"❌ Error: {str(e)}"

    history = history + [
        {"role": "user",      "content": question},
        {"role": "assistant", "content": full_response}
    ]
    return history, "", session_id

# ── 7. Clear function — resets memory by generating new session ID ─────────
def clear_chat():
    new_session_id = str(uuid.uuid4())           # new thread = fresh memory
    return [], "", new_session_id

# ── 8. RAGAS evaluation function ──────────────────────────────────────────
CONTEXT_FILES_DIR = r"C:\Users\sohai\Desktop\LAB\Final_Project\Final_Files"

def load_context(filename: str) -> str:
    path = os.path.join(CONTEXT_FILES_DIR, filename)
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def run_ragas_evaluation():
    """Run RAGAS evaluation and return (results_markdown, langsmith_link_markdown)."""
    try:
        data = {
            "question": [
                "What is hidden layer in neural networks?",
                "What is activation function in neural networks?",
                "What is gradient descent in neural networks?",
                "What is neural network?",
                "What is backpropagation?"
            ],
            "answer": [
                "A hidden layer in neural networks refers to the layers of neurons that exist between the input layer and the output layer.",
                "An activation function is a mathematical function that determines the output of a neuron based on its input.",
                "Gradient descent is a method used to minimize the cost function by iteratively adjusting the model's parameters.",
                "A neural network is inspired by the brain, consisting of interconnected neurons that hold numerical values between zero and one.",
                "Backpropagation evaluates the loss function and determines how much each neuron contributed to errors."
            ],
            "contexts": [
                [load_context("f4.txt")],
                [load_context("f1.txt")],
                [load_context("f2.txt")],
                [load_context("f1.txt")],
                [load_context("f8.txt")]
            ],
            "reference": [
                "A hidden layer is any intermediate layer of neurons between the input and output layers.",
                "Activation functions are non-linear mathematical operations applied to a neuron's output.",
                "Gradient descent minimizes loss by iteratively adjusting network weights and biases.",
                "Neural networks are machine learning systems inspired by the human brain.",
                "Backpropagation calculates the gradient of the loss function to adjust weights."
            ]
        }

        ragas_dataset = Dataset.from_dict(data)
        eval_llm   = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
        embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

        results = evaluate(
            ragas_dataset,
            metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
            llm=LangchainLLMWrapper(eval_llm),
            embeddings=LangchainEmbeddingsWrapper(embeddings)
        )

        metric_names = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
        scores = {
            m: float(np.mean(results[m])) if isinstance(results[m], list) else float(results[m])
            for m in metric_names
        }

        # ── Log aggregate scores to LangSmith ─────────────────────────
        langsmith_client = Client(api_key=os.getenv("LANGSMITH_API_KEY"))
        run_id = uuid.uuid4()
        langsmith_client.create_run(
            id=run_id,
            name="ragas-evaluation",
            run_type="chain",
            project_name="ragas-rag-eval",
            inputs={
                "dataset_size": len(ragas_dataset),
                "metrics": metric_names,
                "model": "gpt-4o-mini"
            },
            outputs=scores,
            start_time=datetime.datetime.utcnow(),
            end_time=datetime.datetime.utcnow()
        )

        # ── Build per-question markdown table ─────────────────────────
        df = results.to_pandas()
        numeric_cols = list(df.select_dtypes(include="number").columns)
        display_cols = ["question"] + numeric_cols if "question" in df.columns else numeric_cols
        table_md = df[display_cols].to_markdown(index=False)

        summary_md = f"""## ✅ Evaluation Complete

| Metric | Score |
|---|---|
| 🎯 Faithfulness | **{scores['faithfulness']:.4f}** |
| 🔍 Answer Relevancy | **{scores['answer_relevancy']:.4f}** |
| 📌 Context Precision | **{scores['context_precision']:.4f}** |
| 📚 Context Recall | **{scores['context_recall']:.4f}** |

---

### 📊 Per-Question Breakdown

{table_md}

---
✅ Results logged to LangSmith — project: `ragas-rag-eval`
"""
        link_md = "🔗 [View full traces on LangSmith](https://smith.langchain.com/projects/ragas-rag-eval)"
        return summary_md, link_md

    except Exception as e:
        return f"❌ Evaluation failed:\n\n```\n{str(e)}\n```", ""

# ── 9. Gradio UI ──────────────────────────────────────────────────────────
with gr.Blocks(
    theme=gr.themes.Soft(),
    title="🧠 Neural Networks RAG Assistant",
    css="""
    #chatbot  { height: 520px; }
    #send-btn { min-width: 100px; }
    footer    { display: none !important; }
    """
) as demo:

    # One unique session ID per browser tab
    session_id = gr.State(value=lambda: str(uuid.uuid4()))

    gr.Markdown(
        """
        # 🧠 Neural Networks RAG Assistant
        Chat with your knowledge base or run a full **RAGAS evaluation** to measure quality.
        All traces are logged to **LangSmith**.
        """
    )

    with gr.Tabs():

        # ── Tab 1 : Chat ───────────────────────────────────────────────
        with gr.Tab("💬 Chat"):

            chatbot = gr.Chatbot(
                elem_id="chatbot",
                type="messages",
                show_copy_button=True,
                bubble_full_width=False,
                avatar_images=(
                    None,
                    "https://em-content.zobj.net/source/twitter/376/robot_1f916.png"
                ),
                label="Conversation",
            )

            with gr.Row():
                question_box = gr.Textbox(
                    placeholder="Ask a question about neural networks...",
                    show_label=False,
                    scale=8,
                    lines=1,
                    autofocus=True,
                )
                send_btn = gr.Button("Send 🚀", variant="primary", scale=1, elem_id="send-btn")

            with gr.Row():
                clear_btn = gr.Button("🗑️ Clear Chat", variant="secondary")
                gr.Markdown("_Each browser tab has its own memory. Clearing resets the conversation._")

            gr.Examples(
                examples=[
                    "What is a neural network?",
                    "Explain backpropagation.",
                    "What are activation functions and why are they used?",
                    "What is a hidden layer?",
                    "How does gradient descent work?",
                    "What is the difference between CNN and RNN?",
                ],
                inputs=question_box,
                label="💡 Example Questions",
            )

            send_btn.click(
                fn=chat,
                inputs=[question_box, chatbot, session_id],
                outputs=[chatbot, question_box, session_id],
            )
            question_box.submit(
                fn=chat,
                inputs=[question_box, chatbot, session_id],
                outputs=[chatbot, question_box, session_id],
            )
            clear_btn.click(
                fn=clear_chat,
                inputs=[],
                outputs=[chatbot, question_box, session_id],
            )

        # ── Tab 2 : RAGAS Evaluation ───────────────────────────────────
        with gr.Tab("📊 RAGAS Evaluation"):

            gr.Markdown(
                """
                ### Evaluate RAG Pipeline Quality
                Runs **5 pre-defined questions** through the pipeline and scores them on:
                - 🎯 **Faithfulness** — are answers grounded in the retrieved context?
                - 🔍 **Answer Relevancy** — how relevant is the answer to the question?
                - 📌 **Context Precision** — is the retrieved context precise?
                - 📚 **Context Recall** — does the context cover the reference answer?

                Results are automatically logged to **LangSmith**.
                """
            )

            eval_btn = gr.Button("▶️ Run RAGAS Evaluation", variant="primary", size="lg")

            eval_results   = gr.Markdown("_Click **Run RAGAS Evaluation** to start._")
            langsmith_link = gr.Markdown()

            eval_btn.click(
                fn=run_ragas_evaluation,
                inputs=[],
                outputs=[eval_results, langsmith_link],
            )

# ── 10. Launch ────────────────────────────────────────────────────────────
if __name__ == "__main__":
    demo.launch(
        server_name="0.0.0.0",   # bind all interfaces — Docker / cloud ready
        share=False,              # set True for a temporary public gradio.live URL
        show_error=True,
    )

C:\Users\sohai\AppData\Local\Temp\ipykernel_26776\2984893907.py:11: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
C:\Users\sohai\AppData\Local\Temp\ipykernel_26776\2984893907.py:11: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
C:\Users\sohai\AppData\Local\Temp\ipykernel_26776\2984893907.py:11: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: f

* Running on local URL:  http://0.0.0.0:7861


Task exception was never retrieved
future: <Task finished name='Task-356' coro=<Server.serve() done, defined at c:\Users\sohai\Desktop\LAB\Final_Project\venv\Lib\site-packages\uvicorn\server.py:67> exception=SystemExit(1)>
Traceback (most recent call last):
  File "c:\Users\sohai\Desktop\LAB\Final_Project\venv\Lib\site-packages\uvicorn\server.py", line 162, in startup
    server = await loop.create_server(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "C:\Users\sohai\anaconda3\Lib\asyncio\base_events.py", line 1630, in create_server
    raise OSError(err.errno, msg) from None
OSError: [Errno 10048] error while attempting to bind on address ('0.0.0.0', 7860): [winerror 10048] only one usage of each socket address (protocol/network address/port) is normally permitted

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\sohai\anaconda3\Lib\threading.py", line 1043, in _bootstrap_inner
    


To create a public link, set `share=True` in `launch()`.


C:\Users\sohai\AppData\Local\Temp\ipykernel_26776\2984893907.py:157: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  llm=LangchainLLMWrapper(eval_llm),
C:\Users\sohai\AppData\Local\Temp\ipykernel_26776\2984893907.py:158: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  embeddings=LangchainEmbeddingsWrapper(embeddings)
Evaluating:   5%|▌         | 1/20 [00:05<01:35,  5.02s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 